# Inline train preprocess: churn from DAC MVP

Эта тетрадка повторяет логику `src/cvm_model/train/preprocess.py`, но разложена по ячейкам, чтобы её можно было запускать и отлаживать в Jupyter пошагово.

Основная идея:

1. Берём DAC-аудиторию в базовом месяце.
2. Размечаем таргет ухода из DAC в следующем месяце.
3. Собираем полный набор фичей через `utils.load_features`.
4. Проверяем missing features, выбросы, target distribution.
5. Опционально сохраняем датасет в S3 `input`.

## 0. Path and env

Если проект не установлен в kernel, `sys.path` позволит импортировать `cvm_model` из локальной папки `src`.

Если есть `.env`, раскомментируй `%dotenv`.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/underplums/Documents/work/organic-return-dac')
src_path = project_root / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

project_root, src_path

In [ ]:
# %load_ext dotenv
# %dotenv /Users/underplums/Documents/work/organic-return-dac/.env

## 1. Imports

In [ ]:
from datetime import datetime
from pathlib import Path

import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import magpie.sql_utils as su

from cvm_model.io import State
import cvm_model.utils as utils
import cvm_model.sql_my as sql
from cvm_model.parameters import (
    features_for_outliers,
    aud_table,
    fav_omni_features_table,
    preperiod_months,
    uplift_rate_prefix,
    aud_suffix,
    features,
    input_suffix,
    template,
)

## 2. Runtime flags

`WRITE_TO_S3 = False` позволяет сначала собрать датасет локально в памяти и не трогать S3. Когда всё проверено, можно поставить `True`.

In [ ]:
event_timestamp = datetime(2025, 11, 1)

WRITE_TO_S3 = False
CLEAN_S3_INPUT = False
USE_TEMP_AUD_TABLE = True
CLEANUP_TEMP_OBJECTS = False

# Для быстрой отладки можно включить sample после загрузки аудитории.
# Для полного прогона поставь None.
SAMPLE_N = 10_000

base_month = event_timestamp.date().replace(day=1).isoformat()
target_month = (pd.Timestamp(base_month) + pd.DateOffset(months=1)).date().isoformat()
feature_date = target_month

base_month, target_month, feature_date

## 3. State and connections

Если падает здесь, значит kernel не видит нужные env-переменные или инфраструктурные доступы.

In [ ]:
state = State.from_env()
engine = state.credentials.loyalty_gp.sa_engine
session = state.spark.session
s3 = su.get_s3_client()

engine, session

## 4. Resolve S3 input path

In [ ]:
input_prefix = state.settings.get_prefix(temp=True, suffix=input_suffix)
input_bucket = input_prefix.split('//')[1].split('/')[0]
input_key = '/'.join(input_prefix.split('//')[1].split('/')[1:])
input_path = template.format(bucket=input_bucket, prefix=input_key)

input_bucket, input_key, input_path

In [ ]:
if CLEAN_S3_INPUT:
    su.remove_from_s3(input_key, dry_run=False)
    files = su.list_all_s3_objects(s3, input_key)
    assert len(files) == 0, 'Не очистилась папка для датасета!'
else:
    print('Skip S3 input cleanup')

## 5. Audience

Аудитория: клиенты, которые являются DAC в `base_month` по новой логике.

In [ ]:
aud_query_raw = sql.aud_query.format(base_month=base_month)

df = utils.get_df(engine, aud_query_raw).fillna(0)
df = df.astype({col: np.int32 for col in {'contact_id'} & set(df.columns)})

assert len(df) > 0
assert 'contact_id' in df.columns
assert df['contact_id'].nunique() == len(df)

print(df.shape)
df.head()

In [ ]:
if SAMPLE_N is not None and len(df) > SAMPLE_N:
    df = df.sample(SAMPLE_N, random_state=42).reset_index(drop=True)
    print('Sampled audience:', df.shape)

df.head()

## 6. Upload audience to temp GP table

Так же делает исходный preprocess: временная таблица ускоряет последующие SQL-запросы.

In [ ]:
if USE_TEMP_AUD_TABLE:
    utils.upload_df(engine, pd.DataFrame(df['contact_id']).astype(int), aud_table)
    aud_query = f'select * from {aud_table}'
else:
    aud_query = aud_query_raw

aud_query[:500]

## 7. Recency

Recency оставляем как фичу. Не фильтруем по `login_recency`, потому что новая DAC-логика включает PWA и offline virtual card.

In [ ]:
query_kwargs = dict(
    aud=aud_query,
    date=feature_date,
    month=preperiod_months[0],
)

df_part = utils.get_df(engine, sql.recency_query.format(**query_kwargs))
assert len(df_part) / len(df) >= 0.99, 'Не выгрузилась recency-таблица более чем для 1% аудитории.'

df = df.merge(df_part, on='contact_id', how='left')

print(df.shape)
df[['contact_id', 'cheque_recency', 'login_recency', 'omni_qr_recency', 'omni_features_recency']].head()

## 8. Refresh temp audience table

Повторяем исходный паттерн: после возможных фильтров обновляем таблицу аудитории.

In [ ]:
if USE_TEMP_AUD_TABLE:
    utils.upload_df(engine, pd.DataFrame(df['contact_id']).astype(int), aud_table)
    aud_query = f'select * from {aud_table}'

aud_query[:500]

## 9. Target

`target_churn_from_dac = 1`: клиент был DAC в базовом месяце и не DAC в следующем.

In [ ]:
query_kwargs = dict(
    aud=aud_query,
    target_month=target_month,
)

df_part = utils.get_df(engine, sql.target_query.format(**query_kwargs)).fillna(0)
df_part = df_part.astype({col: np.int32 for col in {'target_trns', 'target_login'} & set(df_part.columns)})
df_part['target_dac'] = df_part['target_trns'] * df_part['target_login']

df = df.merge(df_part, on='contact_id')

print(df.shape)
display(df['target_churn_from_dac'].value_counts(dropna=False).to_frame('cnt'))
display(df['target_churn_from_dac'].value_counts(normalize=True, dropna=False).to_frame('share'))
df.head()

## 10. Full feature loading

Это тот же `utils.load_features`, что использует исходный preprocess. Здесь будут тяжёлые запросы.

In [ ]:
aud_prefix = state.settings.get_prefix(temp=True, suffix=aud_suffix)

df = utils.load_features(
    engine=engine,
    session=session,
    df=df,
    aud_query=aud_query,
    date=feature_date,
    fav_omni_features_table=fav_omni_features_table,
    preperiod_months=preperiod_months,
    aud_prefix=aud_prefix,
    uplift_rate_prefix=uplift_rate_prefix,
)

print(df.shape)
df.head()

## 11. Feature checks

In [ ]:
missing_features = sorted(set(features) - set(df.columns))
missing_features

In [ ]:
assert len(missing_features) == 0, f'В датасете не хватает фичей: {missing_features}'

display(df[features + ['target_churn_from_dac']].isna().mean().sort_values(ascending=False).to_frame('null_share'))
display(df[features].describe().T)

## 12. Outliers

Исходный preprocess удалял выбросы внутри churn-сегментов. У новой DAC-аудитории этих сегментов нет, поэтому считаем 0.999-квантиль по всей аудитории.

In [ ]:
q = 0.999
df['outlier'] = 0

for f in features_for_outliers:
    if f not in df.columns or df[f].notna().sum() == 0:
        continue
    t = np.nanquantile(df[f], q)
    df.loc[df[f] > t, 'outlier'] = 1

outlier_share = len(df[df['outlier'] == 1]) / len(df)
outlier_share

In [ ]:
assert outlier_share <= 0.01, 'Удалено как выбросы более 1% аудитории.'

df = df[df['outlier'] == 0].reset_index(drop=True)
df = df.drop(columns=['outlier'])

df.shape

## 13. Save local debug parquet

In [ ]:
local_path = project_root / 'df_churn_from_dac_debug.parquet'
df.to_parquet(local_path)
local_path

## 14. Optional S3 save

Включай только когда датасет проверен.

In [ ]:
if WRITE_TO_S3:
    print(f'Saving dataset to {input_bucket}/{input_key}')
    su.save_to_s3(df, input_key, input_type='df', bucket=input_bucket)
else:
    print('Skip S3 save')

## 15. Optional cleanup

In [ ]:
if CLEANUP_TEMP_OBJECTS:
    su.remove_from_s3(aud_prefix, dry_run=False)
    for table in [aud_table, fav_omni_features_table]:
        utils.execute_query(engine, f'drop table if exists {table}')
else:
    print('Skip cleanup')